# Threat Intelligence PESTLEM Analysis

Interactive chat-based threat intelligence analysis using the PESTLEM framework.

## Quick Start

1. **Run Cells 1-2** (setup - one time)
2. **Run Cell 3** to display the chat interface
3. **Type your query** and click "Analyze"
4. Watch the analysis stream in real-time

**PESTLEM Framework:**
- **P**olitical | **E**conomic | **S**ocial | **T**echnical | **L**egal | **E**nvironmental | **M**ilitary

---
# SETUP (Run Cells 1-2 Once)
---
## Cell 1: Install Dependencies & Initialize Environment

In [1]:
# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================
!pip install -q feedparser requests beautifulsoup4 tqdm python-dateutil lxml markdown pdfplumber

import os, re, json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Optional, Tuple
import feedparser, requests, markdown, pdfplumber
from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from tqdm.notebook import tqdm
from IPython.display import HTML, display, Markdown, clear_output

# =============================================================================
# ENVIRONMENT DETECTION
# =============================================================================
IN_COLAB = False
AI_AVAILABLE = False
DRIVE_MOUNTED = False

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# Initialize Colab AI
if IN_COLAB:
    try:
        from google.colab import ai
        AI_AVAILABLE = True
        print("[OK] Google Colab AI: Available")
    except ImportError:
        print("[!!] Google Colab AI: Not available")

    # Mount Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_MOUNTED = True
        print("[OK] Google Drive: Mounted")
    except Exception as e:
        print(f"[!!] Google Drive: {e}")
else:
    print("[!!] Not running in Google Colab - AI features unavailable")

print(f"\nEnvironment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("="*50)
print("Setup complete. Run Cell 2 to configure sources.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 684.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 29.2 MB/s eta 0:00:00
[OK] Google Colab AI: Available
Mounted at /content/drive
[OK] Google Drive: Mounted

Environment: Google Colab
Setup complete. Run Cell 2 to configure sources.


---
## Cell 2: Configuration & Core Functions

In [2]:
# =============================================================================
# ANALYST CONFIGURATION
# =============================================================================
ANALYST_NAME = "TI Agent"
RAG_FOLDER = "/content/drive/MyDrive/Area/threat_intel"
LOOKBACK_DAYS = 5

# =============================================================================
# RSS FEEDS WITH PESTLEM MAPPING AND NATO ADMIRALTY SOURCE RELIABILITY
#
# admiralty_source: NATO AJP-2.1 Source Reliability Grade
#   A = Completely Reliable    B = Usually Reliable     C = Fairly Reliable
#   D = Not Usually Reliable   E = Unreliable           F = Cannot Be Judged
#
# Justifications:
#   A  — FBI Stories: Official U.S. government law enforcement; absolute institutional accountability.
#   B  — Established outlets with named editorial responsibility, track records of accuracy, and
#         independent verification processes (Krebs, DFIR Report, Microsoft MSTIC, BBC, FT, NPR,
#         Risky Business, Cyberscoop, Bellingcat, USAFacts, Healthcare Dive).
#   C  — Trade press, aggregators, or sources with occasional vendor/editorial influence
#         (Security Affairs, CSO Online, RFI, Fierce Healthcare, New Scientist, The Verge,
#          The Guardian tech, Databreaches.net).
#   D  — Sources with structural bias or significant editorial independence concerns
#         (Geopolitical Economy — openly ideological; SCMP — Alibaba ownership since 2016).
# =============================================================================
RSS_FEEDS = [
    # Cyber/Technical sources
    {"name": "Krebs on Security",        "url": "http://krebsonsecurity.com/feed",
     "category": "cyber",                "admiralty_source": "B", "pestlem": ["Technical"]},
    {"name": "Security Affairs",         "url": "http://securityaffairs.com/rss",
     "category": "cyber",                "admiralty_source": "C", "pestlem": ["Technical"]},
    {"name": "The DFIR Report",          "url": "https://thedfirreport.com/feed",
     "category": "cyber",                "admiralty_source": "B", "pestlem": ["Technical"]},
    {"name": "Microsoft Security",       "url": "https://www.microsoft.com/en-us/security/blog/topic/threat-intelligence/feed/",
     "category": "cyber",                "admiralty_source": "B", "pestlem": ["Technical"]},
    {"name": "Risky Business",           "url": "https://risky.biz/rss.xml",
     "category": "cyber",                "admiralty_source": "B", "pestlem": ["Technical", "Political"]},
    {"name": "CSO Online",               "url": "https://www.csoonline.com/feed/",
     "category": "cyber",                "admiralty_source": "C", "pestlem": ["Technical", "Legal"]},
    {"name": "Cyberscoop",               "url": "https://cyberscoop.com/feed/",
     "category": "cyber",                "admiralty_source": "B", "pestlem": ["Technical"]},

    # Government/Political sources
    {"name": "FBI Stories",              "url": "https://www.fbi.gov/news/stories/RSS",
     "category": "government",           "admiralty_source": "B", "pestlem": ["Political", "Legal", "Military"]},
    {"name": "USAFacts",                 "url": "https://usafacts.org/rss/",
     "category": "government",           "admiralty_source": "B", "pestlem": ["Political", "Economic", "Social"]},
      {"name": "USA Congress",                 "url": "https://www.govinfo.gov/rss/bills.xml",
     "category": "government",           "admiralty_source": "B", "pestlem": ["Political","Legal"]},

    # Geopolitical sources
    {"name": "Bellingcat",               "url": "https://www.bellingcat.com/feed/",
     "category": "geopolitical",         "admiralty_source": "B", "pestlem": ["Political", "Military"]},
    {"name": "Geopolitical Economy",     "url": "https://geopoliticaleconomy.com/feed",
     "category": "geopolitical",         "admiralty_source": "D", "pestlem": ["Political", "Economic", "Military"]},
    {"name": "South China Morning Post", "url": "https://www.scmp.com/rss/3/feed",
     "category": "geopolitical",         "admiralty_source": "C", "pestlem": ["Political", "Economic", "Military"]},
    {"name": "RFI International",        "url": "https://www.rfi.fr/en/rss",
     "category": "geopolitical",         "admiralty_source": "C", "pestlem": ["Political", "Social"]},
    {"name": "RAND Research",        "url": "https://www.rand.org/news/press.xml",
     "category": "geopolitical",         "admiralty_source": "B", "pestlem": ["Political", "Social"]},

    # Healthcare/Social sources
    {"name": "Fierce Healthcare",        "url": "https://www.fiercehealthcare.com/rss/regulatory/xml",
     "category": "healthcare",           "admiralty_source": "C", "pestlem": ["Social", "Legal", "Environmental"]},
    {"name": "Healthcare Dive",          "url": "https://www.healthcaredive.com/feeds/news/",
     "category": "healthcare",           "admiralty_source": "C", "pestlem": ["Social", "Economic"]},
    {"name": "BBC Health",               "url": "http://feeds.bbci.co.uk/news/health/rss.xml",
     "category": "healthcare",           "admiralty_source": "B", "pestlem": ["Social", "Environmental"]},

    # General news (multi-PESTLEM)
    {"name": "BBC News",                 "url": "http://feeds.bbci.co.uk/news/rss.xml",
     "category": "general_news",         "admiralty_source": "B", "pestlem": ["Political", "Economic", "Social"]},
    {"name": "NPR Top Stories",          "url": "https://feeds.npr.org/1002/rss.xml",
     "category": "general_news",         "admiralty_source": "B", "pestlem": ["Political", "Social", "Environmental"]},
    {"name": "Financial Times",          "url": "https://www.ft.com/rss/home",
     "category": "general_news",         "admiralty_source": "B", "pestlem": ["Economic", "Political"]},
    {"name": "News 24/680",          "url": "https://news24-680.com/feed/",
     "category": "general_news",         "admiralty_source": "B", "pestlem": ["Social"]},

    # Tech sources
    {"name": "The Verge",                "url": "http://www.theverge.com/rss/index.xml",
     "category": "tech",                 "admiralty_source": "C", "pestlem": ["Technical", "Social"]},
    {"name": "New Scientist",            "url": "https://feeds.newscientist.com/home",
     "category": "tech",                 "admiralty_source": "C", "pestlem": ["Technical", "Environmental"]},
    {"name": "The Guardian",             "url": "https://www.theguardian.com/us/technology/rss",
     "category": "tech",                 "admiralty_source": "C", "pestlem": ["Technical", "Social"]},


    # Data privacy/breach sources
    {"name": "Databreaches.net",         "url": "https://databreaches.net/feed/",
     "category": "cyber",                "admiralty_source": "C", "pestlem": ["Legal", "Technical"]},
]

# =============================================================================
# PUBLISHER INDEPENDENCE MAP
# Feeds sharing the same group value are NOT independent of each other.
# Used by check_corroboration() to prevent same-publisher double-counting.
# =============================================================================
FEED_PUBLISHER_GROUPS = {
    "Krebs on Security":         "krebs",
    "Security Affairs":          "security_affairs",
    "The DFIR Report":           "dfir_report",
    "Microsoft Security":        "microsoft",
    "Risky Business":            "risky_biz",
    "CSO Online":                "idg",
    "FBI Stories":               "us_federal_gov",
    "USAFacts":                  "usafacts",
    "Bellingcat":                "bellingcat",
    "Geopolitical Economy":      "geopolitical_economy",
    "South China Morning Post":  "scmp",
    "RFI International":         "rfi",
    "Fierce Healthcare":         "fierce_media",
    "Healthcare Dive":           "industry_dive",
    "BBC Health":                "bbc",
    "BBC News":                  "bbc",      # Same publisher as BBC Health
    "NPR Top Stories":           "npr",
    "Financial Times":           "ft",
    "The Verge":                 "vox_media",
    "New Scientist":             "new_scientist",
    "The Guardian":              "guardian",
    "Databreaches.net":          "databreaches",
}

# =============================================================================
# NATO ADMIRALTY SYSTEM (AJP-2.1)
# Two-axis framework: Source Reliability (A-F) × Information Credibility (1-6)
# Escalation threshold: B2 and above
# =============================================================================
ADMIRALTY_SOURCE = {
    "A": {"label": "Completely Reliable",   "color": "#15803d", "bg": "#dcfce7", "border": "#15803d"},
    "B": {"label": "Usually Reliable",      "color": "#1d4ed8", "bg": "#dbeafe", "border": "#1d4ed8"},
    "C": {"label": "Fairly Reliable",       "color": "#92400e", "bg": "#fef3c7", "border": "#b45309"},
    "D": {"label": "Not Usually Reliable",  "color": "#c2410c", "bg": "#ffedd5", "border": "#ea580c"},
    "E": {"label": "Unreliable",            "color": "#b91c1c", "bg": "#fee2e2", "border": "#b91c1c"},
    "F": {"label": "Cannot Be Judged",      "color": "#6b7280", "bg": "#f3f4f6", "border": "#9ca3af"},
}

ADMIRALTY_INFO = {
    1: {"label": "Confirmed",                "color": "#15803d", "bg": "#dcfce7", "border": "#15803d"},
    2: {"label": "Probably True",            "color": "#1d4ed8", "bg": "#dbeafe", "border": "#1d4ed8"},
    3: {"label": "Possibly True",            "color": "#92400e", "bg": "#fef3c7", "border": "#b45309"},
    4: {"label": "Doubtful",                 "color": "#c2410c", "bg": "#ffedd5", "border": "#ea580c"},
    5: {"label": "Improbable",               "color": "#b91c1c", "bg": "#fee2e2", "border": "#b91c1c"},
    6: {"label": "Truth Cannot Be Judged",   "color": "#6b7280", "bg": "#f3f4f6", "border": "#9ca3af"},
}

# =============================================================================
# PESTLEM DEFINITIONS
# =============================================================================
PESTLEM = {
    "Political":    "Government policies, regulations, political stability, international relations, elections, governance",
    "Economic":     "Market conditions, financial factors, trade policies, inflation, employment, supply chains",
    "Social":       "Demographics, cultural factors, public opinion, social movements, education, health trends",
    "Technical":    "Technology developments, cybersecurity, infrastructure, innovation, digital transformation",
    "Legal":        "Laws, compliance, legal precedents, regulatory changes, litigation, data protection",
    "Environmental":"Climate change, sustainability, natural resources, environmental regulations, disasters",
    "Military":     "Defense capabilities, security forces, military operations, conflicts, defense spending",
}

# =============================================================================
# WEP SCALE (ICD 203)
# =============================================================================
WEP_SCALE = {
    "remote":             {"min": 0,   "max": 5,   "confidence": "Very Low"},
    "highly unlikely":    {"min": 5,   "max": 20,  "confidence": "Low"},
    "unlikely":           {"min": 20,  "max": 45,  "confidence": "Low to Moderate"},
    "roughly even chance":{"min": 45,  "max": 55,  "confidence": "Moderate"},
    "likely":             {"min": 55,  "max": 80,  "confidence": "Moderate to High"},
    "highly likely":      {"min": 80,  "max": 95,  "confidence": "High"},
    "almost certain":     {"min": 95,  "max": 100, "confidence": "Very High"},
}

def get_wep_term(score):
    for term, v in WEP_SCALE.items():
        if v["min"] <= score < v["max"]: return {"term": term, "confidence": v["confidence"]}
    return {"term": "almost certain", "confidence": "Very High"} if score >= 100 else {"term": "unknown", "confidence": "Unknown"}

# =============================================================================
# CORE FUNCTIONS
# =============================================================================
def fetch_feed(feed_info):
    """Fetch and parse a single RSS feed."""
    try:
        feed = feedparser.parse(feed_info["url"])
        if feed.bozo and not feed.entries: return [], f"Failed: {feed_info['name']}"
        articles = []
        for e in feed.entries:
            pub = None
            for df in ['published_parsed','updated_parsed','created_parsed']:
                if hasattr(e,df) and getattr(e,df):
                    try: pub = datetime(*getattr(e,df)[:6]); break
                    except: pass
            if not pub:
                for df in ['published','updated','created']:
                    if hasattr(e,df) and getattr(e,df):
                        try: pub = date_parser.parse(getattr(e,df)); break
                        except: pass
            if not pub: pub = datetime.now()
            summary = e.get('summary', e.get('description', ''))
            content = e.content[0].get('value', summary) if hasattr(e, 'content') and e.content else summary
            if content: content = BeautifulSoup(content, 'lxml').get_text(' ', strip=True)
            if summary: summary = BeautifulSoup(summary, 'lxml').get_text(' ', strip=True)
            articles.append({
                "title":           e.get('title','No Title'),
                "link":            e.get('link',''),
                "summary":         summary[:500]  if summary else '',
                "content":         content[:1500] if content else '',
                "published":       pub,
                "source":          feed_info["name"],
                "category":        feed_info["category"],
                "admiralty_source":feed_info.get("admiralty_source", "F"),
                "pestlem_tags":    feed_info.get("pestlem", []),
            })
        return articles, None
    except Exception as ex: return [], f"Error: {feed_info['name']}: {ex}"

def fetch_feeds(feeds, workers=10, progress_callback=None):
    """Fetch multiple RSS feeds in parallel."""
    all_articles, errors = [], []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(fetch_feed, f): f for f in feeds}
        for i, fut in enumerate(as_completed(futures)):
            arts, err = fut.result()
            if arts: all_articles.extend(arts)
            if err:  errors.append(err)
            if progress_callback:
                progress_callback(i + 1, len(feeds))
    return all_articles, errors

def smart_filter_articles(articles, days=7, keywords=None, min_results=5):
    """
    Smart article filtering with tokenized matching and fallback.

    Strategy:
    1. Filter by date (required)
    2. Tokenize keywords into individual words
    3. Score articles by keyword word matches
    4. Include articles matching ANY keyword word
    5. Fallback to recent articles if results too few

    Returns:
        tuple: (filtered_articles, diagnostic_info)
    """
    cutoff = datetime.now() - timedelta(days=days)
    date_filtered = [a for a in articles if a["published"] >= cutoff]

    diagnostic = {
        "total_articles":    len(articles),
        "after_date_filter": len(date_filtered),
        "keywords_received": keywords or [],
        "keyword_tokens":    set(),
        "matched_count":     0,
        "fallback_used":     False,
    }

    if not keywords or not date_filtered:
        return sorted(date_filtered, key=lambda x: x["published"], reverse=True), diagnostic

    stop_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
        'should', 'may', 'might', 'must', 'shall', 'can', 'to', 'of', 'in',
        'for', 'on', 'with', 'at', 'by', 'from', 'as', 'into', 'through',
        'during', 'before', 'after', 'above', 'below', 'between', 'under',
        'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where',
        'why', 'how', 'all', 'each', 'few', 'more', 'most', 'other', 'some',
        'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than',
        'too', 'very', 'just', 'and', 'but', 'if', 'or', 'because', 'until',
        'while', 'what', 'which', 'who', 'whom', 'this', 'that', 'these',
        'those', 'am', 'its', 'it', 'new',
    }

    keyword_words = set()
    for kw in keywords:
        words = re.findall(r'\b[a-z]{3,}\b', kw.lower())
        keyword_words.update(w for w in words if w not in stop_words)

    diagnostic["keyword_tokens"] = keyword_words

    if not keyword_words:
        return sorted(date_filtered, key=lambda x: x["published"], reverse=True), diagnostic

    scored = []
    for article in date_filtered:
        text = f"{article['title']} {article['summary']}".lower()
        matches = sum(1 for w in keyword_words if w in text)
        if matches > 0:
            scored.append((article, matches))

    scored.sort(key=lambda x: (-x[1], -x[0]['published'].timestamp()))
    matched = [a for a, _ in scored]
    diagnostic["matched_count"] = len(matched)

    if len(matched) < min_results:
        diagnostic["fallback_used"] = True
        unmatched = [a for a in date_filtered if a not in matched]
        unmatched.sort(key=lambda x: x["published"], reverse=True)
        matched.extend(unmatched[:min_results - len(matched)])

    return matched, diagnostic

def filter_articles(articles, days=7, keywords=None):
    """Legacy filter — delegates to smart_filter_articles."""
    result, _ = smart_filter_articles(articles, days, keywords)
    return result

# =============================================================================
# NATO ADMIRALTY SYSTEM — CORROBORATION & RATING FUNCTIONS
# =============================================================================
def check_corroboration(article, all_articles, min_word_overlap=3):
    """
    Count how many INDEPENDENT publishers corroborate this article's story.

    AJP-2.1 requires corroboration from genuinely independent sources.
    Feeds sharing the same FEED_PUBLISHER_GROUPS value are treated as the
    same publisher and cannot corroborate each other (e.g. BBC News + BBC Health).

    Args:
        article:          The article being evaluated.
        all_articles:     Pool of articles to check against (typically date-filtered set).
        min_word_overlap: Significant title words in common to count as covering same story.

    Returns:
        dict:
            independent_count (int):        Number of distinct independent publishers corroborating.
            corroborating_sources (list):   Source names that corroborate.
    """
    _STOP = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on', 'at', 'to',
        'for', 'of', 'and', 'or', 'but', 'with', 'by', 'from', 'as', 'this',
        'that', 'it', 'its', 'be', 'has', 'have', 'had', 'will', 'new', 'how',
        'why', 'what', 'when', 'where', 'who', 'not', 'no', 'up', 'out', 'says',
        'said', 'after', 'over', 'into', 'than', 'more', 'just', 'about',
    }

    def _tokens(text):
        words = re.findall(r'\b[a-z]{4,}\b', text.lower())
        return {w for w in words if w not in _STOP}

    target_words  = _tokens(article.get('title', ''))
    target_source = article.get('source', '')
    target_group  = FEED_PUBLISHER_GROUPS.get(target_source, target_source)

    if not target_words:
        return {"independent_count": 0, "corroborating_sources": []}

    seen_groups  = set()
    seen_sources = set()

    for other in all_articles:
        if other is article:
            continue
        other_source = other.get('source', '')
        other_group  = FEED_PUBLISHER_GROUPS.get(other_source, other_source)

        # Skip same publisher or already counted
        if other_group == target_group or other_group in seen_groups:
            continue

        overlap = target_words & _tokens(other.get('title', ''))
        if len(overlap) >= min_word_overlap:
            seen_groups.add(other_group)
            seen_sources.add(other_source)

    return {
        "independent_count":     len(seen_groups),
        "corroborating_sources": sorted(seen_sources),
    }

def assign_admiralty_rating(article, corroboration_result):
    """
    Assign NATO Admiralty System (AJP-2.1) rating to an article.

    Source Reliability (A-F) is fixed per feed (admiralty_source field).
    Information Credibility (1-6) is computed dynamically from corroboration
    count and article recency — the two axes are evaluated independently
    per AJP-2.1 doctrine.

    Escalation threshold: source A or B AND info grade 1 or 2.

    Information credibility assignment rules:
        ≥2 independent sources corroborate → 1 (Confirmed)
        A/B source + 1 corroboration OR A/B + breaking (<24h) → 2 (Probably True)
        C source + 1 corroboration OR A/B source uncorroborated/older → 3 (Possibly True)
        C source uncorroborated OR D + any corroboration → 4 (Doubtful)
        D/E source uncorroborated → 5 (Improbable)
        F or unknown → 6 (Truth Cannot Be Judged)

    Returns:
        Article dict enriched with admiralty_* and wep_* fields.
    """
    source_grade      = article.get('admiralty_source', 'F')
    pub_time          = article.get('published', datetime.now())
    is_recent_24h     = (datetime.now() - pub_time).total_seconds() < 86400
    independent_count = corroboration_result.get('independent_count', 0)

    # --- Assign information credibility (1-6) ---
    if independent_count >= 2:
        info_grade = 1
    elif independent_count == 1 and source_grade in ('A', 'B'):
        info_grade = 2
    elif source_grade in ('A', 'B') and is_recent_24h:
        info_grade = 2   # Breaking news from reliable source
    elif independent_count == 1 and source_grade == 'C':
        info_grade = 3
    elif source_grade in ('A', 'B'):
        info_grade = 3   # Reliable but uncorroborated, not breaking
    elif source_grade == 'C':
        info_grade = 4
    elif independent_count >= 1 and source_grade == 'D':
        info_grade = 4
    elif source_grade in ('D', 'E'):
        info_grade = 5
    else:
        info_grade = 6   # F or unknown

    rating = f"{source_grade}{info_grade}"

    # Escalation: B2 or better (source A or B, info 1 or 2)
    _src_rank = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6}
    above_escalation = _src_rank.get(source_grade, 6) <= 2 and info_grade <= 2

    # WEP compatibility — map info credibility to estimative language
    _info_to_wep = {
        1: ("almost certain",    "Very High"),
        2: ("highly likely",     "High"),
        3: ("likely",            "Moderate to High"),
        4: ("unlikely",          "Low to Moderate"),
        5: ("highly unlikely",   "Low"),
        6: ("unknown",           "Unknown"),
    }
    wep_term, wep_confidence = _info_to_wep.get(info_grade, ("unknown", "Unknown"))

    return {
        **article,
        "admiralty_source":       source_grade,
        "admiralty_info":         info_grade,
        "admiralty_rating":       rating,
        "admiralty_source_label": ADMIRALTY_SOURCE[source_grade]["label"],
        "admiralty_info_label":   ADMIRALTY_INFO[info_grade]["label"],
        "above_escalation":       above_escalation,
        "corroboration":          corroboration_result,
        "wep_term":               wep_term,
        "wep_confidence":         wep_confidence,
        "trust_score":            None,  # Replaced by admiralty_rating
    }

def admiralty_sort_key(article):
    """
    Sort key tuple (source_rank, info_rank) — lower = higher priority.
    A1 → (1,1) highest, F6 → (6,6) lowest. Source is primary sort axis.
    """
    _rank = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6}
    return (
        _rank.get(article.get('admiralty_source', 'F'), 6),
        article.get('admiralty_info', 6),
    )

def render_admiralty_badge(article):
    """
    Render a two-pill HTML badge showing source reliability letter | info credibility number.
    Appends an ESCALATE flag when the article meets the B2 threshold.
    """
    s      = article.get('admiralty_source', 'F')
    i      = article.get('admiralty_info', 6)
    above  = article.get('above_escalation', False)
    corr   = article.get('corroboration', {})
    c_cnt  = corr.get('independent_count', 0)
    s_meta = ADMIRALTY_SOURCE.get(s, ADMIRALTY_SOURCE['F'])
    i_meta = ADMIRALTY_INFO.get(i, ADMIRALTY_INFO[6])

    escalation_html = (
        ' <span style="background:#fbbf24;color:#78350f;border:1px solid #d97706;'
        'border-radius:3px;padding:1px 5px;font-size:11px;font-weight:700;'
        'font-family:Arial,sans-serif;letter-spacing:0.4px;">ESCALATE</span>'
        if above else ''
    )
    corr_note = (
        f'Corroborated by {c_cnt} independent source(s)'
        if c_cnt > 0 else 'No independent corroboration'
    )
    return (
        f'<span style="display:inline-flex;align-items:center;font-family:monospace;font-weight:700;font-size:13px;">'
        f'<span style="padding:2px 7px;border-radius:3px 0 0 3px;border:1px solid {s_meta["border"]};'
        f'background:{s_meta["bg"]};color:{s_meta["color"]};" title="{s_meta["label"]}">{s}</span>'
        f'<span style="padding:2px 7px;border-radius:0 3px 3px 0;border:1px solid {i_meta["border"]};'
        f'border-left:1px solid rgba(0,0,0,0.15);background:{i_meta["bg"]};color:{i_meta["color"]};" '
        f'title="{i_meta["label"]}">{i}</span>'
        f'</span>'
        f'{escalation_html}'
        f' <span style="font-size:0.8em;color:#6b7280;font-style:italic;">— {corr_note}</span>'
    )

def score_article(article):
    """Legacy stub: assign admiralty rating with zero corroboration (backward compat)."""
    return assign_admiralty_rating(article, {"independent_count": 0, "corroborating_sources": []})

def load_rag_documents(folder_path, max_chars=5000):
    """Load documents from Google Drive folder (supports .txt, .md, .pdf)."""
    documents = []
    if not folder_path or not os.path.exists(folder_path):
        return documents

    for filename in os.listdir(folder_path):
        ext      = os.path.splitext(filename)[1].lower()
        filepath = os.path.join(folder_path, filename)
        if not os.path.isfile(filepath):
            continue
        content = None
        try:
            if ext in {'.txt', '.md'}:
                with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()[:max_chars]
            elif ext == '.pdf':
                text_parts = []
                with pdfplumber.open(filepath) as pdf:
                    for page in pdf.pages:
                        page_text = page.extract_text()
                        if page_text:
                            text_parts.append(page_text)
                        if sum(len(t) for t in text_parts) >= max_chars:
                            break
                content = "\n\n".join(text_parts)[:max_chars]
            if content:
                documents.append({'filename': filename, 'content': content})
        except Exception as e:
            print(f"[Warning] Could not read {filename}: {e}")

    return documents

def get_relevant_feeds(pestlem_dims):
    """Get RSS feeds relevant to specified PESTLEM dimensions."""
    return [f for f in RSS_FEEDS if any(d in f.get("pestlem", []) for d in pestlem_dims)]

def generate_html_report(query, analysis, rated_articles, selected_feeds,
                         relevant_dims, analyst_name, escalation_count=0):
    """
    Generate HTML report with NATO Admiralty System source ratings and legend.
    Articles are sorted by Admiralty priority (A1 first, F6 last).
    """
    try:
        analysis_html = markdown.markdown(analysis, extensions=['tables', 'fenced_code'])
    except:
        analysis_html = analysis.replace('\n', '<br>')

    dt       = datetime.now().strftime('%B %d, %Y %H:%M')
    dims_str = ', '.join(relevant_dims)

    # Admiralty legend tables
    source_legend = "".join([
        f'<tr>'
        f'<td style="font-weight:700;background:{v["bg"]};color:{v["color"]};'
        f'padding:3px 8px;border-radius:3px;text-align:center;border:1px solid {v["border"]};">{k}</td>'
        f'<td style="padding:3px 8px;">{v["label"]}</td>'
        f'</tr>'
        for k, v in ADMIRALTY_SOURCE.items()
    ])
    info_legend = "".join([
        f'<tr>'
        f'<td style="font-weight:700;background:{v["bg"]};color:{v["color"]};'
        f'padding:3px 8px;border-radius:3px;text-align:center;border:1px solid {v["border"]};">{k}</td>'
        f'<td style="padding:3px 8px;">{v["label"]}</td>'
        f'</tr>'
        for k, v in ADMIRALTY_INFO.items()
    ])

    # Article list sorted by Admiralty priority
    top_articles = sorted(rated_articles, key=admiralty_sort_key)[:10]
    article_items = []
    for a in top_articles:
        badge    = render_admiralty_badge(a)
        title    = a['title'][:80]
        date_str = a['published'].strftime('%Y-%m-%d') if isinstance(a.get('published'), datetime) else ''
        article_items.append(
            f'<li style="margin:8px 0;">{badge}&nbsp;'
            f'<b>{title}...</b><br>'
            f'<span style="font-size:0.85em;color:#555;">'
            f'{a["source"]} | {date_str} | '
            f'Source: {a.get("admiralty_source_label","")} | '
            f'Info: {a.get("admiralty_info_label","")}'
            f'</span></li>'
        )
    article_list_html = '\n'.join(article_items)

    escalation_banner = (
        f'<div style="background:#fef3c7;border:1px solid #d97706;border-radius:5px;'
        f'padding:8px 15px;margin:10px 0;font-weight:700;color:#78350f;">'
        f'⚡ {escalation_count} article(s) meet the B2 escalation threshold — priority action warranted.'
        f'</div>'
    ) if escalation_count > 0 else ''

    html_report = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>PESTLEM Intelligence Assessment</title>
    <style>
        body {{ font-family: Arial, sans-serif; line-height: 1.6; color: #333; max-width: 950px; margin: 0 auto; padding: 20px; }}
        h1   {{ border-bottom: 3px solid #7c3aed; padding-bottom: 10px; color: #7c3aed; }}
        h2   {{ color: #7c3aed; margin-top: 25px; border-bottom: 1px solid #ddd; padding-bottom: 5px; }}
        h3   {{ color: #0066cc; margin-top: 20px; }}
        .header    {{ background: #f5f3ff; padding: 15px; border-left: 4px solid #7c3aed; margin: 20px 0; }}
        .query     {{ background: #fffbf0; padding: 15px; border-left: 4px solid #ffa500; margin: 20px 0; font-style: italic; }}
        .analysis  {{ background: #fafafa; padding: 20px; border: 1px solid #ddd; border-radius: 8px; margin: 20px 0; }}
        .sources   {{ background: #f0f8ff; padding: 15px; border: 1px solid #0066cc; margin: 20px 0; font-size: 0.9em; }}
        .adm-legend {{ display: flex; gap: 30px; margin: 12px 0; flex-wrap: wrap; }}
        .adm-legend table {{ border-collapse: separate; border-spacing: 3px; font-size: 0.85em; }}
        .classification {{ text-align: center; font-weight: bold; color: #22c55e; padding: 10px; background: #f0f0f0; }}
        .footer {{ margin-top: 30px; padding-top: 15px; border-top: 1px solid #ddd; font-size: 0.85em; color: #666; }}
        ul {{ margin: 10px 0; padding-left: 25px; }}
        li {{ margin: 5px 0; }}
        @media print {{ body {{ font-size: 11pt; }} .analysis {{ break-inside: avoid; }} }}
    </style>
</head>
<body>
    <div class="classification">UNCLASSIFIED</div>

    <h1>PESTLEM Intelligence Assessment</h1>

    <div class="header">
        <b>Date:</b> {dt}<br>
        <b>Analyst:</b> {analyst_name}<br>
        <b>Dimensions:</b> {dims_str}<br>
        <b>Sources:</b> {len(rated_articles)} articles from {len(selected_feeds)} feeds<br>
        <b>Escalation Findings (B2 or above):</b> {escalation_count}<br>
        <b>Source Grading:</b> NATO Admiralty System (AJP-2.1)<br>
        <b>AI Model:</b> Google Gemini (via google.colab.ai)
    </div>

    <h2>Intelligence Query</h2>
    <div class="query">{query.strip()}</div>

    <h2>PESTLEM Analysis</h2>
    <div class="analysis">{analysis_html}</div>

    <h2>Sources Consulted</h2>
    <div class="sources">
        <b>RSS Feeds:</b> {', '.join([f['name'] for f in selected_feeds])}<br><br>

        <b>NATO Admiralty System Rating Legend (AJP-2.1):</b>
        <div class="adm-legend">
            <div><b>Source Reliability</b><table>{source_legend}</table></div>
            <div><b>Information Credibility</b><table>{info_legend}</table></div>
        </div>

        {escalation_banner}

        <b>Top Articles by Admiralty Priority:</b>
        <ul>{article_list_html}</ul>
    </div>

    <div class="footer">
        Generated by PESTLEM Intelligence Analysis Notebook<br>
        Framework: Political, Economic, Social, Technical, Legal, Environmental, Military<br>
        Source Grading: NATO Admiralty System AJP-2.1 | Confidence Language: ICD 203 Words of Estimative Probability
    </div>

    <div class="classification">UNCLASSIFIED</div>
</body>
</html>'''
    return html_report


print(f"Analyst:              {ANALYST_NAME}")
print(f"RAG Folder:           {RAG_FOLDER if RAG_FOLDER else 'Not configured'}")
print(f"Lookback:             {LOOKBACK_DAYS} days")
print(f"RSS Feeds:            {len(RSS_FEEDS)} sources configured")
print(f"Source Grading:       NATO Admiralty System (AJP-2.1)")
print(f"Escalation Threshold: B2 or above")
print("="*55)
print("Configuration complete. Run Cell 3 to launch the chat interface.")

Analyst:              TI Agent
RAG Folder:           /content/drive/MyDrive/Area/threat_intel
Lookback:             5 days
RSS Feeds:            26 sources configured
Source Grading:       NATO Admiralty System (AJP-2.1)
Escalation Threshold: B2 or above
Configuration complete. Run Cell 3 to launch the chat interface.


---
---
# ANALYSIS INTERFACE
---
## Cell 3: Interactive Chat Interface

Type your intelligence query below and click **Analyze** to run the full PESTLEM analysis pipeline.

In [3]:
# =============================================================================
# INTERACTIVE CHAT INTERFACE FOR PESTLEM ANALYSIS
# =============================================================================
# =============================================================================
# ENVIRONMENT GUARD — initializes Colab globals if cell-2 was skipped
# =============================================================================
if 'AI_AVAILABLE' not in globals():
    IN_COLAB      = False
    AI_AVAILABLE  = False
    DRIVE_MOUNTED = False
    colab_files   = None
    try:
        from google.colab import files as colab_files
        IN_COLAB = True
        try:
            from google.colab import ai
            AI_AVAILABLE = True
        except ImportError:
            pass
    except ImportError:
        pass
    if not AI_AVAILABLE:
        print("[System] Warning: AI_AVAILABLE=False. "
              "Run cell-2 first, or ensure google.colab is available.")

from ipywidgets import Textarea, Button, VBox, HBox, Output, HTML as WidgetHTML, Layout
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# =============================================================================
# GLOBAL STATE
# =============================================================================
analysis_state = {
    'last_report_html': None,
    'last_report_md':   None,
    'html_filename':    None,
    'md_filename':      None,
    'is_running':       False,
}

# =============================================================================
# UI COMPONENTS
# =============================================================================
chat_header = WidgetHTML(value="""
<div style="background: linear-gradient(135deg, #7c3aed 0%, #5b21b6 100%);
            color: white; padding: 20px; border-radius: 10px 10px 0 0; margin-bottom: 0;">
    <h2 style="margin: 0; font-weight: 600;">PESTLEM Intelligence Analysis</h2>
    <p style="margin: 5px 0 0 0; opacity: 0.9; font-size: 14px;">Enter your threat intelligence query below</p>
</div>
""")

chat_history = Output(layout=Layout(
    border='1px solid #e5e7eb',
    min_height='350px',
    max_height='500px',
    overflow_y='auto',
    padding='15px',
    background_color='#fafafa',
    border_radius='0px'
))

query_input = Textarea(
    placeholder='Enter your threat intelligence question here...\n\nExample: What are the emerging cyber threats to critical infrastructure from state-sponsored actors?',
    layout=Layout(width='100%', height='100px', border='1px solid #d1d5db', border_radius='0px')
)

analyze_btn = Button(
    description='Analyze',
    button_style='success',
    icon='search',
    layout=Layout(width='120px')
)

clear_btn = Button(
    description='Clear History',
    button_style='warning',
    icon='trash',
    layout=Layout(width='130px')
)

download_btn = Button(
    description='Download Report',
    button_style='info',
    icon='download',
    layout=Layout(width='150px'),
    disabled=True
)

status_output = Output()

# =============================================================================
# ANALYSIS PIPELINE
# =============================================================================
def run_analysis(query):
    """Run the complete PESTLEM analysis pipeline with NATO Admiralty System source grading."""
    global analysis_state

    if not AI_AVAILABLE:
        print("[ERROR] Google Colab AI not available.")
        print("This notebook requires Google Colab with google.colab.ai.")
        return

    print(f"\n{'='*60}")
    print(f"[Analyst] {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*60}")
    print(f"{query}\n")

    # Step 1: Load RAG documents FIRST (primary source)
    rag_documents = []
    rag_context   = ""
    rag_filenames = []
    if RAG_FOLDER and DRIVE_MOUNTED:
        print(f"[System] Loading RAG documents...")
        rag_documents = load_rag_documents(RAG_FOLDER)
        if rag_documents:
            print(f"[System] Loaded {len(rag_documents)} reference documents:")
            for doc in rag_documents:
                print(f"         - {doc['filename']} ({len(doc['content'])} chars)")
                rag_filenames.append(doc['filename'])
            rag_context = "\n\n".join([f"--- {d['filename']} ---\n{d['content']}" for d in rag_documents])

    # Step 2: Analyze query for PESTLEM dimensions and keywords
    print(f"[System] Analyzing query for PESTLEM dimensions...")

    analysis_prompt = f"""Analyze this intelligence query and provide a structured response.

QUERY: {query}

PESTLEM FRAMEWORK:
- Political: {PESTLEM['Political']}
- Economic: {PESTLEM['Economic']}
- Social: {PESTLEM['Social']}
- Technical: {PESTLEM['Technical']}
- Legal: {PESTLEM['Legal']}
- Environmental: {PESTLEM['Environmental']}
- Military: {PESTLEM['Military']}

Respond in this EXACT JSON format (no markdown, just raw JSON):
{{
  "relevant_dimensions": ["Dimension1", "Dimension2"],
  "primary_dimension": "MostRelevantDimension",
  "keywords": ["keyword1", "keyword2", "keyword3"],
  "analysis_focus": "Brief description of what to focus on",
  "time_horizon": "immediate/short-term/medium-term/long-term"
}}

IMPORTANT KEYWORD RULES:
1. Provide 8-12 search keywords total
2. Include single important words (e.g., "breach", "privacy", "regulation")
3. Include geographic terms SEPARATELY
4. Include synonyms for key concepts
5. Keep multi-word phrases SHORT (2-3 words max)
6. Focus on nouns, key verbs, and proper nouns
7. Split compound concepts

Select 2-4 most relevant PESTLEM dimensions."""

    try:
        analysis_response = ai.generate_text(analysis_prompt)
        json_match = re.search(r'\{[^{}]*\}', analysis_response, re.DOTALL)
        if json_match:
            query_analysis = json.loads(json_match.group())
        else:
            query_analysis = {
                "relevant_dimensions": ["Technical", "Political"],
                "primary_dimension":   "Technical",
                "keywords":            ["cyber", "threat", "attack", "security"],
                "analysis_focus":      "General threat analysis",
                "time_horizon":        "short-term",
            }
    except Exception as e:
        print(f"[System] AI analysis error: {e}")
        query_analysis = {
            "relevant_dimensions": ["Technical", "Political"],
            "primary_dimension":   "Technical",
            "keywords":            ["cyber", "threat", "security"],
            "analysis_focus":      "General threat analysis",
            "time_horizon":        "short-term",
        }

    relevant_dims   = query_analysis.get('relevant_dimensions', ['Technical', 'Political'])
    search_keywords = query_analysis.get('keywords', [])

    print(f"[System] Focus: {', '.join(relevant_dims)}")
    print(f"[System] Keywords: {', '.join(search_keywords)}")

    # Step 3: Fetch RSS feeds
    selected_feeds = get_relevant_feeds(relevant_dims)
    print(f"\n[System] Fetching from {len(selected_feeds)} relevant sources...")

    def progress_update(current, total):
        pct = int((current / total) * 100)
        bar = '=' * (pct // 5) + '>' + '.' * (20 - pct // 5)
        print(f"\r[System] Progress: [{bar}] {pct}%", end='', flush=True)

    all_articles, errors = fetch_feeds(selected_feeds, progress_callback=progress_update)
    print(f"\n[System] Retrieved {len(all_articles)} articles")

    # Step 4: Filter articles by date and keywords
    filtered_articles, filter_diagnostic = smart_filter_articles(all_articles, LOOKBACK_DAYS, search_keywords)

    print(f"[System] Filtering Diagnostics:")
    print(f"         - Total articles: {filter_diagnostic['total_articles']}")
    print(f"         - After date filter ({LOOKBACK_DAYS} days): {filter_diagnostic['after_date_filter']}")
    print(f"         - Keyword tokens: {', '.join(sorted(filter_diagnostic['keyword_tokens'])) if filter_diagnostic['keyword_tokens'] else 'None'}")
    print(f"         - Articles matched by keywords: {filter_diagnostic['matched_count']}")
    if filter_diagnostic['fallback_used']:
        print(f"         - Fallback: Added recent articles to meet minimum threshold")

    # Step 5: NATO Admiralty System — corroboration check and rating assignment
    # Each article's Source Reliability (A-F) comes from its feed configuration.
    # Information Credibility (1-6) is assigned independently based on corroboration
    # across the filtered article pool and article recency (AJP-2.1 doctrine).
    print(f"\n[System] Running corroboration analysis ({len(filtered_articles)} articles)...")

    rated_articles = []
    for article in filtered_articles:
        corr  = check_corroboration(article, filtered_articles)
        rated = assign_admiralty_rating(article, corr)
        rated_articles.append(rated)

    escalation_count = sum(1 for a in rated_articles if a.get('above_escalation', False))

    # Admiralty rating distribution summary
    adm_dist = {}
    for a in rated_articles:
        r = a.get('admiralty_rating', 'F6')
        adm_dist[r] = adm_dist.get(r, 0) + 1
    dist_summary = ', '.join(f'{r}:{c}' for r, c in sorted(adm_dist.items()))

    print(f"[System] Admiralty ratings assigned: {len(rated_articles)} articles")
    print(f"         Escalation findings (B2 or above): {escalation_count}")
    print(f"         Distribution: {dist_summary}")

    # Step 6: Build analysis context — articles sorted by Admiralty priority (A1 first)
    article_context = ""
    top_articles    = []
    if rated_articles:
        top_articles = sorted(rated_articles, key=admiralty_sort_key)[:15]
        article_parts = []
        for i, art in enumerate(top_articles, 1):
            date_str = art['published'].strftime('%Y-%m-%d') if isinstance(art['published'], datetime) else str(art['published'])[:10]
            esc_flag = " [ESCALATE]" if art.get('above_escalation', False) else ""
            corr_cnt = art.get('corroboration', {}).get('independent_count', 0)
            article_parts.append(
                f"[{i}] {art['title']}\n"
                f"    Source: {art['source']} | Date: {date_str}\n"
                f"    Admiralty: {art.get('admiralty_rating','F6')}{esc_flag}"
                f" ({art.get('admiralty_source_label','')} / {art.get('admiralty_info_label','')})\n"
                f"    Corroboration: {corr_cnt} independent source(s)\n"
                f"    PESTLEM: {', '.join(art.get('pestlem_tags', []))}\n"
                f"    Summary: {art.get('summary', '')[:250]}"
            )
        article_context = "\n\n".join(article_parts)

    dimension_desc = "\n".join([f"- **{dim}**: {PESTLEM[dim]}" for dim in relevant_dims])

    # Step 7: Generate PESTLEM analysis
    print(f"\n[System] Generating PESTLEM analysis...\n")
    print("-" * 60)

    admiralty_instructions = """3. Apply NATO Admiralty System (AJP-2.1) source weighting:
   - Articles labelled [ESCALATE] are rated B2 or above — treat as priority intelligence
   - Source Reliability: A=Completely Reliable, B=Usually Reliable, C=Fairly Reliable, D=Not Usually Reliable
   - Information Credibility: 1=Confirmed, 2=Probably True, 3=Possibly True, 4=Doubtful, 5=Improbable
   - Weight your analysis toward A1/B1/B2 rated articles; apply proportional skepticism to C4 and below
   - Note where key claims rely solely on low-rated sources"""

    if rag_context:
        source_instructions = f"""## INSTRUCTIONS
1. **PRIORITY: First analyze the REFERENCE DOCUMENTS section below** — these are uploaded intelligence reports that are PRIMARY sources for this analysis
2. If the query mentions a specific document by name, base your analysis primarily on that document's content
{admiralty_instructions}
4. Use Words of Estimative Probability (WEP) per ICD 203:
   - Almost Certain (95-100%), Highly Likely (80-95%), Likely (55-80%)
   - Roughly Even Chance (45-55%), Unlikely (20-45%), Highly Unlikely (5-20%), Remote (0-5%)
5. Use the CURRENT INTELLIGENCE (RSS articles) to supplement and provide additional context
6. Cite specific findings from the reference documents when applicable
7. Note intelligence gaps"""
    else:
        source_instructions = f"""## INSTRUCTIONS
1. Analyze the query through each specified PESTLEM dimension
2. Base your analysis on the provided intelligence articles
{admiralty_instructions}
4. Use Words of Estimative Probability (WEP) per ICD 203:
   - Almost Certain (95-100%), Highly Likely (80-95%), Likely (55-80%)
   - Roughly Even Chance (45-55%), Unlikely (20-45%), Highly Unlikely (5-20%), Remote (0-5%)
5. Provide 3-5 sentences of specific analysis for each dimension
6. Include actionable insights
7. Note intelligence gaps"""

    gen_prompt = f"""You are a senior intelligence analyst conducting a PESTLEM assessment.

## ANALYST QUERY
{query}

## PESTLEM DIMENSIONS TO ANALYZE
{dimension_desc}

{source_instructions}

## FORMAT
For each dimension:
### [Dimension Name]
[Your analysis with WEP language]

End with:
### Key Takeaways
[3-5 bullet points]

### Intelligence Gaps
[Areas needing more information]

### Recommended Actions
[2-3 actionable recommendations]
"""

    # RAG documents first (primary source)
    if rag_context:
        gen_prompt += f"\n\n## REFERENCE DOCUMENTS (PRIMARY SOURCE)\nThe following documents were uploaded by the analyst as primary reference material.\n\n{rag_context[:15000]}"

    # RSS articles as supplementary context
    if article_context:
        gen_prompt += (
            f"\n\n## CURRENT INTELLIGENCE ({len(top_articles)} articles, sorted by Admiralty priority)\n"
            f"Articles marked [ESCALATE] meet the B2 threshold per NATO AJP-2.1.\n\n"
            f"{article_context}"
        )

    pestlem_analysis = ""
    try:
        stream = ai.generate_text(gen_prompt, stream=True)
        for chunk in stream:
            print(chunk, end='', flush=True)
            pestlem_analysis += chunk
    except Exception as e:
        print(f"\n[ERROR] Analysis generation failed: {e}")
        pestlem_analysis = f"Analysis generation failed: {e}"

    print("\n" + "-" * 60)

    # Step 8: Generate and save reports
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    html_report = generate_html_report(
        query, pestlem_analysis, rated_articles,
        selected_feeds, relevant_dims, ANALYST_NAME,
        escalation_count=escalation_count
    )

    html_filename = f"pestlem_report_{timestamp}.html"
    with open(html_filename, 'w', encoding='utf-8') as f:
        f.write(html_report)

    md_content = f"""# PESTLEM Intelligence Assessment

**Date:** {datetime.now().strftime('%B %d, %Y %H:%M')}
**Analyst:** {ANALYST_NAME}
**Dimensions:** {', '.join(relevant_dims)}
**Source Grading:** NATO Admiralty System (AJP-2.1)
**Escalation Findings (B2+):** {escalation_count}

## Query
{query}

---

{pestlem_analysis}

---
*Generated by PESTLEM Intelligence Analysis Notebook*
*Source grading per NATO Admiralty System AJP-2.1 | Confidence language per ICD 203*
"""
    md_filename = f"pestlem_report_{timestamp}.md"
    with open(md_filename, 'w', encoding='utf-8') as f:
        f.write(md_content)

    analysis_state['last_report_html'] = html_report
    analysis_state['last_report_md']   = md_content
    analysis_state['html_filename']    = html_filename
    analysis_state['md_filename']      = md_filename

    print(f"\n[System] Report saved:    {html_filename}")
    print(f"[System] Markdown saved:  {md_filename}")

    # Report preview
    print(f"\n{'='*60}")
    print("REPORT PREVIEW")
    print(f"{'='*60}")
    display(HTML(html_report))

    print(f"\n[System] Analysis complete! Use 'Download Report' to save files.")

# =============================================================================
# EVENT HANDLERS
# =============================================================================
def on_analyze_click(b):
    """Handle analyze button click."""
    global analysis_state

    if analysis_state['is_running']:
        return

    query = query_input.value.strip()
    if not query:
        with chat_history:
            print("[System] Please enter a query before analyzing.")
        return

    analysis_state['is_running'] = True
    analyze_btn.disabled         = True
    analyze_btn.description      = 'Analyzing...'
    download_btn.disabled        = True

    with chat_history:
        try:
            run_analysis(query)
            download_btn.disabled = False
        except Exception as e:
            print(f"\n[ERROR] {e}")
        finally:
            analysis_state['is_running'] = False
            analyze_btn.disabled         = False
            analyze_btn.description      = 'Analyze'

def on_clear_click(b):
    """Handle clear button click."""
    chat_history.clear_output()
    with chat_history:
        print("[System] Chat history cleared. Enter a new query to begin.")

def on_download_click(b):
    """Handle download button click."""
    if analysis_state['html_filename'] and IN_COLAB:
        try:
            colab_files.download(analysis_state['html_filename'])
            colab_files.download(analysis_state['md_filename'])
            with chat_history:
                print(f"\n[System] Downloading reports...")
        except Exception as e:
            with chat_history:
                print(f"\n[System] Download error: {e}")
    elif not IN_COLAB:
        with chat_history:
            print(f"\n[System] Files saved locally:")
            print(f"  - {analysis_state['html_filename']}")
            print(f"  - {analysis_state['md_filename']}")

# Connect event handlers
analyze_btn.on_click(on_analyze_click)
clear_btn.on_click(on_clear_click)
download_btn.on_click(on_download_click)

# =============================================================================
# DISPLAY INTERFACE
# =============================================================================
button_bar = HBox(
    [analyze_btn, clear_btn, download_btn],
    layout=Layout(
        padding='10px',
        background_color='#f3f4f6',
        border='1px solid #e5e7eb',
        border_radius='0 0 10px 10px'
    )
)

chat_container = VBox(
    [chat_header, chat_history, query_input, button_bar],
    layout=Layout(
        border='1px solid #e5e7eb',
        border_radius='10px',
        box_shadow='0 4px 6px rgba(0, 0, 0, 0.1)'
    )
)

# Initial message
with chat_history:
    print("[System] PESTLEM Intelligence Analysis ready.")
    print(f"[System] Analyst:              {ANALYST_NAME}")
    print(f"[System] Sources:              {len(RSS_FEEDS)} RSS feeds configured")
    print(f"[System] Lookback:             {LOOKBACK_DAYS} days")
    print(f"[System] Source Grading:       NATO Admiralty System (AJP-2.1)")
    print(f"[System] Escalation Threshold: B2 or above")
    if RAG_FOLDER:
        print(f"[System] RAG:                  {RAG_FOLDER}")
    print("\nEnter your intelligence query and click 'Analyze' to begin.")

display(chat_container)


[Analyst] 16:42:31
What are three top threats Handl Health, a post-series A healthcare technology company building AI-powered care navigation and cost estimation products should consider?

[System] Loading RAG documents...
[System] Loaded 41 reference documents:
         - Microsoft digital defense report_2023.pdf (5000 chars)
         - 2023-netdiligence-cyber-claims-study.pdf (5000 chars)
         - 2025 Google threat intel team adversarial-misuse-generative-ai.pdf (5000 chars)
         - 2024-dbir-data-breach-investigations-report (2).pdf (5000 chars)
         - 2024-dbir-data-breach-investigations-report (1).pdf (5000 chars)
         - Agentic-AI-Threats-and-Mitigations_v1.0a.pdf (5000 chars)
         - 2024-dbir-data-breach-investigations-report.pdf (5000 chars)
         - 2022-dbir-data-breach-investigations-report.pdf (5000 chars)
         - 2022-sonicwall-cyber-threat-report.pdf (5000 chars)
         - 2024_IC3Report.pdf (5000 chars)
         - 2025-Cyber-Claims-Study-Report-1

A,Completely Reliable
B,Usually Reliable
C,Fairly Reliable
D,Not Usually Reliable
E,Unreliable
F,Cannot Be Judged
1,Confirmed
2,Probably True
3,Possibly True
4,Doubtful
5,Improbable



[System] Analysis complete! Use 'Download Report' to save files.


---
## Cell 4: Additional Export Options (Optional)

Run this cell for additional export options after completing an analysis.

In [5]:
# =============================================================================
# ADDITIONAL EXPORT OPTIONS
# =============================================================================

if analysis_state.get('last_report_html'):
    print("Last Analysis Report")
    print("="*50)
    print(f"HTML: {analysis_state['html_filename']}")
    print(f"Markdown: {analysis_state['md_filename']}")
    print("\nOptions:")

    # Display download buttons
    display(HTML(f'''
    <div style="padding:15px;background:#f5f3ff;border:1px solid #7c3aed;border-radius:5px;margin:20px 0;">
    <b>Export Options:</b><br><br>
    <button onclick="window.print()" style="padding:10px 20px;background:#7c3aed;color:white;border:none;border-radius:4px;cursor:pointer;margin-right:10px;">Print Report</button>
    </div>
    '''))

    # Re-download files
    if IN_COLAB:
        print("\nDownloading files...")
        colab_files.download(analysis_state['html_filename'])
        colab_files.download(analysis_state['md_filename'])
else:
    print("No analysis completed yet.")
    print("Run Cell 3 and submit a query first.")

Last Analysis Report
HTML: pestlem_report_20260413_164514.html
Markdown: pestlem_report_20260413_164514.md

Options:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>